<a href="https://colab.research.google.com/github/Numanur/llm-practice/blob/main/LLM_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount(
    '/content/drive'
)

Mounted at /content/drive


In [ ]:
!pip install transformers accelerate

In [30]:
import numpy as np
import pandas as pd
from tqdm import tqdm

In [3]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
)

# **Generator configuration for Phi-3**

In [7]:
# from transformers import pipeline

# generator = pipeline(
#     "text-generation",
#     model = model,
#     tokenizer = tokenizer,
#     return_full_text = False,
#     max_new_tokens = 200,
#     do_sample = True
# )

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


# **Configuration for Mistral**

In [13]:
from transformers import pipeline

In [14]:
# tokenizer.pad_token = tokenizer.eos_token

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=200,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id
)

# This cell should be changed depending on the model

In [19]:
messages = [
    # {"role": "system", "content": "If you don't know the answer, say you don't know."},
    {"role": "user", "content": "Tell me about Suman Saha's contribution to quantum oncology."}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

output = generator(prompt)
print(output[0]["generated_text"])

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Suman Saha is a researcher in the field of quantum computing and its applications, particularly in the area of quantum oncology. His work focuses on using quantum computing to develop more effective and personalized cancer treatments.

One of Saha's significant contributions is the development of quantum algorithms for drug discovery and optimization of cancer treatments. He has proposed quantum algorithms for simulating the behavior of molecules, which can help in understanding the interactions between drugs and cancer cells at a molecular level. This level of detail can potentially lead to the discovery of new drugs and the optimization of existing ones.

Saha has also worked on quantum algorithms for radiation therapy planning. Radiation therapy is a common treatment for cancer, but it can be challenging to deliver the right dose of radiation to the tumor while minimizing damage to healthy cells. Quantum computing could potentially help optimize radiation therapy plans, making trea

# **Dense Retrieval "Semantic Search" for RAG**

In [ ]:
!pip install cohere

In [29]:
import cohere

api_key = 'BxCtQ0QTet1dFvpLfT3rfk7pPNPsiIHATZheTjpb'
co =  cohere.Client(api_key)

NameError: name 'cohere' is not defined

In [20]:
text = """
Interstellar is a 2014 epic science fiction film co-written,
directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain,
Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to
survive, the film follows a group of astronauts who travel
through a wormhole near Saturn in search of a new home for
mankind.
Brothers Christopher and Jonathan Nolan wrote the screenplay,
which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in
Physics[4] Kip Thorne was an executive producer, acted as a
scientific consultant, and wrote a tie-in book, The Science of
Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in
the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in
Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and
the company Double Negative created additional digital effects.
Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock,
expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773
million with subsequent re-releases), making it the tenth-highest
grossing film of 2014.
It received acclaim for its performances, direction, screenplay,
musical score, visual effects, ambition, themes, and emotional
weight.
It has also received praise from many astronomers for its
scientific accuracy and portrayal of theoretical astrophysics.
Since its premiere, Interstellar gained a cult following,[5] and
now is regarded by many sci-fi experts as one of the best
science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy
Awards, winning Best Visual Effects, and received numerous other
accolades"""


In [21]:
texts = text.split(".")
texts = [t.strip(' \n') for t in texts]


In [22]:
response = co.embed(
    texts=texts,
    model="embed-v4.0",   # or "embed-v4.0"
    input_type="search_document",
).embeddings

embeds = np.array(response)
print(embeds.shape)

NameError: name 'co' is not defined

In [23]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 60.8 MB/s eta 0:00:00


In [24]:
from sentence_transformers import SentenceTransformer

In [25]:
import faiss
dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim)
print(index.is_trained)
index.add(np.float32(embeds))


NameError: name 'embeds' is not defined

In [ ]:
def search(query, number_of_results=3):
    query_embed = co.embed(
        texts=[query],
        model="embed-v4.0",
        input_type="search_query",
    ).embeddings[0]

    distances, similar_item_ids = index.search(
        np.float32([query_embed]),
        number_of_results
    )

    texts_np = np.array(texts)
    results = pd.DataFrame({
        "texts": texts_np[similar_item_ids[0]],
        "distance": distances[0]
    })

    print(f"Query: {query}\nNearest neighbors:")
    return results

In [ ]:
query = "how far the moon is"
result = search(query)
result

# **Loading local models instead of Cohere**

In [26]:
# 1) Load local embedding model
model = SentenceTransformer("BAAI/bge-base-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [27]:
embeds = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
print(embeds.shape)

(15, 768)


In [31]:
dim = embeds.shape[1]
index = faiss.IndexFlatIP(dim)   # use IP with normalized embeddings for cosine similarity
index.add(np.float32(embeds))

In [32]:
# 5) Search function
def search(query, number_of_results=3):
    query_embed = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, similar_item_ids = index.search(np.float32(query_embed), number_of_results)

    texts_np = np.array(texts)
    results = pd.DataFrame({
        "texts": texts_np[similar_item_ids[0]],
        "score": scores[0]
    })

    print(f"Query: {query}\nNearest neighbors:")
    return results

In [33]:
# 6) Run search
query = "how precise was the science"
result = search(query)
result

Query: how precise was the science
Nearest neighbors:


,texts,score
0,It has also received praise from many astronom...,0.614431
1,Caltech theoretical physicist and 2017 Nobel l...,0.518071
2,"Since its premiere, Interstellar gained a cult...",0.473704


In [34]:
import nbformat

# Load your notebook file
file_path = '/content/drive/MyDrive/Colab Notebooks/LLM_1.ipynb'
with open(file_path, 'r', encoding='utf-8') as f:
    nb = nbformat.read(f, as_version=4)

# Remove the problematic widgets metadata
if 'widgets' in nb.metadata:
    del nb.metadata['widgets']
    print("Metadata 'widgets' removed successfully.")

# Save the clean version
with open('clean_notebook.ipynb', 'w', encoding='utf-8') as f:
    nbformat.write(nb, f)

Metadata 'widgets' removed successfully.
